# CompBench Local + Multi-object → mixed-modality prompts

Loads the official `BohanJia/CompBench` `train` split and selects the Local Editing (`add`, `remove`, `replace`) and Multi-object Editing (`multi_object_add`, `multi_object_remove`) tasks used by this project. Set `RESUME_LAST_RUN = True` in the setup cell to continue the latest annotation run, skipping completed samples and retrying audited failures; otherwise a new timestamped run folder is created.

Install once from the repository root: `uv sync`. Copy `.env.example` to `.env` and set `GEMINI_API_KEY`. CompBench references are grounded only by the dataset-provided masks; no object-grounding model is used. Change `COMPBENCH_TASKS` if you want a different task subset.

In [ ]:
from annotation import (
    AnnotationConfig,
    compbench,
    comma_separated_env,
    create_annotation_run_directory,
    resolve_annotation_run_directory,
    run_hf_annotation,
)

DATASET_ID = "BohanJia/CompBench"
SPLIT = "train"
TASKS = comma_separated_env(
    "COMPBENCH_TASKS",
    "add,remove,replace,multi_object_add,multi_object_remove"
)
CONFIG = AnnotationConfig.from_env("compbench")
RESUME_LAST_RUN = False  # True continues the latest run and skips completed samples.
print("tasks:", sorted(TASKS))

In [ ]:
from dataclasses import replace

RUN_DIR = (
    resolve_annotation_run_directory(CONFIG.output_dir)
    if RESUME_LAST_RUN
    else create_annotation_run_directory(CONFIG.output_dir)
)
RUN_CONFIG = replace(CONFIG, output_dir=RUN_DIR)
print("Annotation run directory:", RUN_DIR)

summary = run_hf_annotation(
    dataset_id=DATASET_ID,
    split=SPLIT,
    adapter=compbench,
    config=RUN_CONFIG,
    output_name="compbench_local_multiobject_mixed.jsonl",
    audit_name="compbench_local_multiobject_audit.jsonl",
    expected_columns={"task", "image_path", "input_image", "edited_image", "instruction", "caption", "mask"},
    use_grounder=False,
    row_filter=lambda row: row["task"] in TASKS,
)
summary

CompBench masks are nullable only for `action`, `location`, and `view`; the selected Local/Multi-object tasks therefore normally carry edit-region masks. All references in this notebook are parsed and grounded from the dataset-provided mask; no text-based object-grounding model is used. `add` uses its edit mask as a spatial destination reference. For multi-object add/remove, connected components (or a projection-based split for touching regions) are parsed independently, with regions merged when they map to the same instruction phrase.